# 1. Importação dos Dados e Bibliotecas.

Nesta etapa, a biblioteca Pandas é importada e o arquivo de assinaturas é carregado e convertido em um DataFrame.

In [1]:
from pathlib import Path
import sys

RAIZ    = Path.cwd().parent
BRONZE  = RAIZ / "data" / "Bronze"
STAGING = RAIZ / "data" / "Staging"
SILVER  = RAIZ / "data" / "Silver"

assert BRONZE.exists(), f"Pasta não encontrada: {BRONZE}. Abra o notebook a partir da pasta notebooks/."

STAGING.mkdir(parents=True, exist_ok=True)
SILVER.mkdir(parents=True, exist_ok=True)
sys.path.append(str(RAIZ))


# Importação de Bibliotecas:
import pandas as pd
import numpy as np

## Importando funções da pasta src.
from src.datas import converter_data

df_interacoes = pd.read_csv(BRONZE / "interacoes.csv")
print(df_interacoes.info())

<class 'pandas.DataFrame'>
RangeIndex: 10015 entries, 0 to 10014
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   id_interacao    10015 non-null  int64
 1   id_cliente      10015 non-null  int64
 2   data_interacao  10015 non-null  str  
 3   tipo            10015 non-null  str  
 4   categoria       10015 non-null  str  
dtypes: int64(2), str(3)
memory usage: 661.6 KB
None


# 2. Tratamento e Análise de Dados:

Nesta etapa, é realizado o tratamento completo dos dados, incluindo a validação de registros duplicados, a alteração de status e a conversão de tipos (types), com base na análise dos seguintes pontos críticos;

## 2.1 Tratamento Coluna id_interacao
### Validação e Exclusão de IDs duplicados.

Nesta etapa, está sendo realizada a análise a respeito se existem dados duplicados, onde devemos considerar como verdadeiro caso seja maior que zero, além disso, foi realizado o filtro para análise dos IDs duplicados, a fim de verificar se trata-se de uma linha duplicada ou não, para seguir com a devida tratativa.

In [2]:
# 1. Validação de IDs duplicados:
total_duplicados = df_interacoes['id_interacao'].duplicated().sum()
print(f"Total de IDs duplicados: {total_duplicados}")

# 2. Tratamento de IDs duplicados: mantendo apenas a primeira ocorrência e removendo as repetições a partir da 2ª vez
df_interacoes = df_interacoes.drop_duplicates(subset=['id_interacao'], keep='first')
total_duplicados = df_interacoes['id_interacao'].duplicated().sum()

# 3. Validação pós tratamento:
print(f"Total de IDs duplicados pós tratamento: {total_duplicados}")


Total de IDs duplicados: 15
Total de IDs duplicados pós tratamento: 0


## 2.2 Tratamento Coluna data_interacao
### Transformação Type DateTime

Nesta etapa, será realizada a transformação da coluna de data_interacao para o formato datatime us, visto que ao analisar o arquivo na camada Bronze, foi identificado que o mesmo está em formato de string.

In [3]:
# 1. Tratamento de Dados: Transformação do formato de datetime, para padronizar para o estilo iso.
df_interacoes['data_interacao'] = converter_data(df_interacoes['data_interacao'])

# 1.2. Validação de Dados: Validando dados Pós Tratamento:
print("Tipo data_interacao:", df_interacoes['data_interacao'].dtype)
print("Datas de interacao vazias:", df_interacoes['data_interacao'].isna().sum())

# 1. Tratamento de Datas futuras:
# 1.2 Validação se existem datas futuras.
hoje = pd.Timestamp.today().normalize()
tem_datas_futuras = (df_interacoes['data_interacao'] >= hoje).any()
print(f'Existem datas posteriores ao dia de hoje ou igual? Resposta: {tem_datas_futuras}')

Tipo data_interacao: datetime64[us]
Datas de interacao vazias: 0
Existem datas posteriores ao dia de hoje ou igual? Resposta: False


## 2.3 Tratamento Coluna tipo
### Análisando valores da Coluna e Tratamento

Nesta etapa, foi feita a análise dos valores contidos na coluna tipo, para identificar divergências de nomes, como por exemplo: escrito de maneira incorreta, valores errados e etc. Após a validação desses dados, foi realizado o tratamento e padronização dos dados, para evitar divergências futuras em relatórios.

Além disso, por regra negócio, em caso de futuros preenchimentos de nomes que não estejam no dicionário de correção, o nome do tipo na qual chegou no data frame, será alteração para o nome: "Investigar", para que a equipe possa análisar a qual tipo pertence aquele dado.

In [4]:
# 1. Identificando valores contidos na coluna para tratamento de dados.
valores_unicos = df_interacoes['tipo'].unique()
print(f'Dados brutos (Sem Tratamento): {valores_unicos}')

# 2. Realizando a correção dos nomes e padronização dos dados:
# 2.1 Removendo todos os espaços extras de dados:
df_interacoes['tipo'] = df_interacoes['tipo'].str.strip()

# 2.2 Criação do dicionario de correção:
tipo_permitidos = {
    'reclamacao': 'Reclamação',
    'Reclamacao': 'Reclamação',
    'RECLAMAÇÃO': 'Reclamação',
    'Duvida': 'Dúvida',
    'duvida': 'Dúvida',
    'cancelamento': 'Cancelamento',
    'CANCELAMENTO': 'Cancelamento',
    'suporte': 'Suporte'
}

# 2.3 Limpeza de Dados: Aplicando o dicionário de correção na minha coluna tipo, para corrigir os valores e manter a padronização.
df_interacoes['tipo'] = df_interacoes['tipo'].replace(tipo_permitidos)

# 2.4 Condicional: Caso entre algum valor que não esteja nessa lista, será modificado para "Investigar"
# O .get() procura a chave; se não achar, retorna "Investigar" como padrão
df_interacoes['tipo'] = df_interacoes['tipo'].apply(lambda x: tipo_permitidos.get(x, 'Investigar') 
    if x not in ['Reclamação', 'Dúvida', 'Cancelamento', 'Suporte', 'Sugestão'] else x)

# 3. Validação de Dados Pós Tratamento:
valores_unicos = df_interacoes['tipo'].unique()
print(f'Pós Tratamento: {valores_unicos}')

Dados brutos (Sem Tratamento): <ArrowStringArray>
[     'Suporte',   'Reclamação',       'Dúvida',     'Sugestão',
 'Cancelamento', ' reclamacao ',       'Duvida',      'suporte',
   'RECLAMAÇÃO']
Length: 9, dtype: str
Pós Tratamento: <ArrowStringArray>
['Suporte', 'Reclamação', 'Dúvida', 'Sugestão', 'Cancelamento']
Length: 5, dtype: str


In [5]:
print(df_interacoes['tipo'].value_counts())

tipo
Suporte         3567
Dúvida          2453
Reclamação      2012
Sugestão        1173
Cancelamento     795
Name: count, dtype: int64


## 2.3 Tratamento Coluna categoria
### Análisando valores da Coluna e Tratamento

Nesta etapa, foi feita a análise dos valores contidos na coluna categoria, para identificar divergências de nomes, como por exemplo: escrito de maneira incorreta, valores errados e etc. Após a validação desses dados, foi realizado o tratamento e padronização dos dados, para evitar divergências futuras em relatórios.

Além disso, por regra negócio, em caso de futuros preenchimentos de nomes que não estejam no dicionário de correção, o nome da categoria na qual chegou no data frame, será alteração para o nome: "Investigar", para que a equipe possa análisar a qual categoria pertence aquele dado.

In [6]:
# 1. Identificando valores contidos na coluna para tratamento de dados.
valores_unicos = df_interacoes['categoria'].unique()
print(f'Dados brutos (Sem Tratamento): {valores_unicos}')

# 2. Realizando a correção dos nomes e padronização dos dados:
# 2.1 Removendo todos os espaços extras de dados:
df_interacoes['categoria'] = df_interacoes['categoria'].str.strip()

# 2.2 Dicionário permitido:
categorias_permitidas = [
    'Cobrança',
    'Produto',
    'Cancelamento',
    'Acesso',
    'Atendimento'
]

# 2.3 Condicional: Caso entre algum valor que não esteja nessa lista, será modificado para "Investigar"
# O .get() procura a chave; se não achar, retorna "Investigar" como padrão
df_interacoes['categoria'] = df_interacoes['categoria'].apply(
    lambda x: x if x in categorias_permitidas else 'Investigar'
)

# 3. Validação de Dados Pós Tratamento:
valores_unicos = df_interacoes['categoria'].unique()
print(f'Pós Tratamento: {valores_unicos}')

Dados brutos (Sem Tratamento): <ArrowStringArray>
['Cobrança', 'Produto', 'Cancelamento', 'Acesso', 'Atendimento']
Length: 5, dtype: str
Pós Tratamento: <ArrowStringArray>
['Cobrança', 'Produto', 'Cancelamento', 'Acesso', 'Atendimento']
Length: 5, dtype: str


# 3. Data Quality & Validação.

Nesta etapa, o DataFrame tratado é submetido a testes de validação antes de ser promovido para a camada Staging (Preparação de Dados).

In [7]:
resultado_qualidade = {}

# 1. Validaçaõ de IDs não nulos:
resultado_qualidade['ID não nulo'] = (
    df_interacoes['id_interacao'].isnull().sum() == 0
)

# 2. Validação a fim de identificar se existem IDs duplicados
resultado_qualidade['ID único'] = (
    df_interacoes['id_interacao'].duplicated().sum() == 0
)

# 3. Validação se existem linhas duplicadas
resultado_qualidade['Sem duplicatas completas'] = (
    df_interacoes.duplicated().sum() == 0
)

# 4. Validação se todas as linhas da coluna id_interacao estão preenchidas
resultado_qualidade['Linhas preenchidas id'] = (
    df_interacoes['id_interacao'].notna().all()
)

# 5. Validação se a coluna data_interacao está com o formato correto de type
resultado_qualidade['Tipo de dado da coluna data_interacao'] = (
    pd.api.types.is_datetime64_any_dtype(df_interacoes['data_interacao'])
)

# 6. Validação valores de IDs negativos
resultado_qualidade['Data interação preenchida'] = (
    df_interacoes['data_interacao'].notna().all()
)

# 7. Validando se restou algum valor indesejado na coluna Tipo:
valores_permitidos_tipo = ['Reclamação', 'Dúvida', 'Cancelamento', 'Suporte', 'Sugestão']

# Atribui o resultado True/False diretamente na coluna do dicionário de qualidade
resultado_qualidade['Sem dados instrusos na coluna tipo'] = (
    df_interacoes['tipo'].isin(valores_permitidos_tipo).all()
)

# 8. Validando se restou algum valor indesejado na coluna Categoria:
valores_permitidos_categoria = ['Cobrança', 'Produto', 'Cancelamento', 'Acesso', 'Atendimento']

# Atribui o resultado True/False diretamente na coluna do dicionário de qualidade
resultado_qualidade['Sem dados instrusos na coluna categoria'] = (
    df_interacoes['categoria'].isin(valores_permitidos_categoria).all()
)

# 9. Validação de datas futuras, visto que não é possível existir interações com datas futuras.

data_atual = pd.Timestamp.today().normalize()
resultado_qualidade['Sem datas futuras'] = (
    (df_interacoes['data_interacao'] <= data_atual).all()
)

# 4. Data Quality: Homologação e Envio para a Camada Staging (Preparação de Dados)

Nesta etapa, o console exibe o resultado das validações realizadas:

Aprovado (True em todos os testes): O DataFrame é liberado e enviado para a camada Staging (Preparação de Dados)
Reprovado (qualquer valor False): O processo é bloqueado, exigindo a correção dos dados antes de avançar.

In [8]:
df_qualidade = pd.DataFrame(
    list(resultado_qualidade.items()),
    columns=['Regra', 'Aprovado']
)

df_qualidade['Status'] = df_qualidade['Aprovado'].map({
    True: 'APROVADO',
    False: 'REPROVADO'
})

display(df_qualidade)

todos_testes_ok = df_qualidade['Aprovado'].all()
caminho_saida = (STAGING / "dim_interacoes.csv")

if todos_testes_ok:
    print("Sucesso: Todos os testes passaram. O arquivo foi enviado para a camada Staging (Preparação de Dados).")

    df_interacoes.to_csv(
        caminho_saida,
        index=False,
        sep=';',
        encoding='utf-8-sig'
    )

else:
    print("Alerta: O arquivo foi impedido de subir para a camada Staging (Preparação de Dados). Existem regras reprovadas.")
    raise ValueError("Falha na validação de qualidade dos dados. Processo interrompido.")

,Regra,Aprovado,Status
0,ID não nulo,True,APROVADO
1,ID único,True,APROVADO
2,Sem duplicatas completas,True,APROVADO
3,Linhas preenchidas id,True,APROVADO
4,Tipo de dado da coluna data_interacao,True,APROVADO
5,Data interação preenchida,True,APROVADO
6,Sem dados instrusos na coluna tipo,True,APROVADO
7,Sem dados instrusos na coluna categoria,True,APROVADO
8,Sem datas futuras,True,APROVADO


Sucesso: Todos os testes passaram. O arquivo foi enviado para a camada Staging (Preparação de Dados).
